# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described using a Croissant schema and is accessible from the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Install `mlcroissant` if not already installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using the Croissant schema with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access full metadata as an object
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}\n")
# (Optional) View further metadata details
# print(json.dumps(metadata.to_json(), indent=2))

## 2. Data Overview

Let's review the available record sets and their associated fields, referencing their respective `@id`s as defined in the Croissant schema.

In [ ]:
# List all available RecordSet @ids and their fields

record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
record_set_ids = []
record_set_fields = {}
for rs in record_sets:
    print(f"Record Set: {rs.id}")
    record_set_ids.append(rs.id)
    if len(rs.fields) > 0:
        print("  Fields and their @ids:")
        field_ids = []
        for field in rs.fields:
            print(f"   - {field.name} \t (field @id: {field.id})")
            field_ids.append(field.id)
        record_set_fields[rs.id] = field_ids
    else:
        print("   (No fields)")
    print()
if not record_sets:
    print("No record sets found in the metadata.")

## 3. Data Extraction

Load data from a selected record set into a DataFrame for data exploration. We will use the `@id` of a record set and its fields as shown above.

If no record sets are present (for example if all data is in a single table), you might need to use the distribution directly or check the Croissant metadata for available data resources.

In [ ]:
# For demonstration, extract data from the first available record set (if present)
dataframes = {}
if len(record_set_ids) > 0:
    for record_set_id in record_set_ids:
        print(f"\nExtracting records for RecordSet: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns in {record_set_id}: \n  {df.columns.tolist()}\n")
    # For illustration, select the first record set
    main_record_set_id = record_set_ids[0]
    print(f"Displaying first few records from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets were found in dataset metadata. Data may be provided directly in distribution.")


## 4. Exploratory Data Analysis (EDA)

Typical EDA operations can include filtering records, normalization, and grouping by relevant fields. Refer to field `@id`s as shown above.

In [ ]:
import numpy as np

# Set your record set and field @ids for EDA
if len(dataframes) > 0:
    # Example for the first record set
    df = dataframes[main_record_set_id]
    print(f"Available columns: {df.columns.tolist()}")
    # Try to auto-detect a numeric field for demo; fallback to user choice
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]  # For demonstration, use first numeric field
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notnull().sum() > 0 else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt grouping by a categorical field
        group_candidates = df.select_dtypes(include=[object]).columns.tolist()
        group_field = group_candidates[0] if len(group_candidates) > 0 else None
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No obvious categorical field for grouping found.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No dataframes loaded. Unable to perform EDA.")

## 5. Visualization

Visualize distributions or relationships between fields using plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the numeric field's distribution
if len(dataframes) > 0 and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
    # If group_field exists and not too many categories, plot as well
    if 'group_field' in locals() and group_field and df[group_field].nunique() < 20:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion

In this notebook, we demonstrated how to:

- Load Croissant metadata and data from a FAIR² dataset using `mlcroissant`.
- List and reference record sets and fields by their unique `@id`.
- Extract data to Pandas dataframes for analysis.
- Apply basic EDA: filtering, normalizing, and grouping numeric fields.
- Visualize data distributions using histograms and boxplots.

You can now build further analyses or apply machine learning workflows on top of this data. For more, see the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).